# Frequently Bought Together by Season and Store

This notebook shows the strongest product combinations across seasons and stores, with both a table and an inline bar chart.

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor

conn = psycopg2.connect(
    host='db',
    port=5432,
    dbname='zava',
    user='postgres',
    password='P@ssw0rd!'
)

sql = '''
WITH order_pairs AS (
    SELECT
        oi1.order_id,
        oi1.store_id,
        CASE
            WHEN oi1.product_id < oi2.product_id THEN oi1.product_id
            ELSE oi2.product_id
        END AS product_a,
        CASE
            WHEN oi1.product_id < oi2.product_id THEN oi2.product_id
            ELSE oi1.product_id
        END AS product_b,
        CASE
            WHEN EXTRACT(MONTH FROM o.order_date) IN (3, 4, 5) THEN 'Spring'
            WHEN EXTRACT(MONTH FROM o.order_date) IN (6, 7, 8) THEN 'Summer'
            WHEN EXTRACT(MONTH FROM o.order_date) IN (9, 10, 11) THEN 'Fall'
            ELSE 'Winter'
        END AS season
    FROM retail.order_items oi1
    JOIN retail.order_items oi2
      ON oi1.order_id = oi2.order_id
     AND oi1.product_id < oi2.product_id
    JOIN retail.orders o
      ON o.order_id = oi1.order_id
),
pair_summary AS (
    SELECT
        op.season,
        s.store_name,
        p1.product_name AS product_a_name,
        p2.product_name AS product_b_name,
        COUNT(*) AS pair_count
    FROM order_pairs op
    JOIN retail.stores s
      ON s.store_id = op.store_id
    JOIN retail.products p1
      ON p1.product_id = op.product_a
    JOIN retail.products p2
      ON p2.product_id = op.product_b
    GROUP BY
        op.season,
        s.store_name,
        p1.product_name,
        p2.product_name
)
SELECT
    season,
    store_name,
    product_a_name,
    product_b_name,
    pair_count
FROM pair_summary
WHERE pair_count >= 5
ORDER BY season, store_name, pair_count DESC;
'''

with conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute(sql, (5,))
    rows = cur.fetchall()

conn.close()

summary = pd.DataFrame(rows)
summary

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

top = summary.head(10)
pairs = [f"{r['product_a_name']} + {r['product_b_name']}" for _, r in top.iterrows()]
counts = top['pair_count'].tolist()

plt.figure(figsize=(12, 7))
bars = plt.bar(pairs, counts, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'])
plt.title('Frequently Bought Together by Season and Store')
plt.xlabel('Product Pair')
plt.ylabel('Pair Count')
plt.xticks(rotation=35, ha='right')
plt.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.5, f'{value}', ha='center', va='bottom')

plt.tight_layout()
plt.show()